# 🔄 Customer Retention Intelligence
## Predicción de Churn + Análisis de Cohortes — Olist Brasil 2016–2018

![Python](https://img.shields.io/badge/Python-3.11-3776AB?logo=python&logoColor=white) ![Pandas](https://img.shields.io/badge/Pandas-2.x-150458?logo=pandas) ![Scikit-learn](https://img.shields.io/badge/Scikit--learn-1.x-F7931E?logo=scikit-learn) ![XGBoost](https://img.shields.io/badge/XGBoost-2.x-EC4E20) ![SHAP](https://img.shields.io/badge/SHAP-Explainability-purple) ![Plotly](https://img.shields.io/badge/Plotly-5.x-3F4F75?logo=plotly) ![Streamlit](https://img.shields.io/badge/Streamlit-Dashboard-FF4B4B?logo=streamlit)

---
# 1. 📋 Contexto del Negocio

## ¿Qué es el Churn y por qué importa?

El **churn** (o abandono de clientes) es una de las métricas más críticas para cualquier empresa. Adquirir un nuevo cliente cuesta entre **5x y 25x más** que retener uno existente.

## El Caso Olist

**Olist** es la plataforma de e-commerce más grande de Brasil, conectando pequeños comercios con los principales marketplaces. El dataset contiene **100,000+ pedidos** realizados entre 2016 y 2018.

## Objetivo

Construir un sistema que:
1. **Prediga con 30 días de anticipación** qué clientes están en riesgo de no volver
2. **Visualice la retención histórica** por cohortes mensuales
3. **Priorice intervenciones** del equipo de Customer Success

---
# 2. 🔍 Exploración de Datos (EDA)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Project color palette
COLORS = {
    "primary": "#9E6240",
    "secondary": "#DEA47E",
    "accent": "#CD4631",
    "base": "#F8F2DC",
    "contrast": "#81ADC8",
    "bg_dark": "#1C1410",
}

import plotly.io as pio
pio.templates["cri"] = pio.templates["plotly_dark"]
pio.templates["cri"].layout.paper_bgcolor = "rgba(0,0,0,0)"
pio.templates["cri"].layout.plot_bgcolor = "#1C1410"
pio.templates["cri"].layout.font = dict(family="Inter", color="#F8F2DC")
pio.templates.default = "cri"


In [ ]:
# Cargar los 9 archivos CSV de Olist
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

print(f'Pedidos: {len(orders):,}')
print(f'Clientes: {len(customers):,}')
print(f'Items: {len(items):,}')
print(f'Pagos: {len(payments):,}')
print(f'Reviews: {len(reviews):,}')

In [ ]:
# Estadísticas descriptivas de los pedidos
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
print('Rango temporal:', orders['order_purchase_timestamp'].min().date(), '→', orders['order_purchase_timestamp'].max().date())
print('Estados de pedido:')
print(orders['order_status'].value_counts())

In [ ]:
# Distribución de pagos
fig = px.histogram(payments, x='payment_value', nbins=50,
                   title='Distribución del Valor de Pagos',
                   color_discrete_sequence=['#DEA47E'])
fig.update_layout(xaxis_title='Valor (R$)', yaxis_title='Frecuencia')
fig.show()

In [ ]:
# Distribución de reviews
fig = px.histogram(reviews, x='review_score', nbins=5,
                   title='Distribución de Puntuaciones de Review',
                   color_discrete_sequence=['#81ADC8'])
fig.update_layout(xaxis_title='Puntuación', yaxis_title='Frecuencia')
fig.show()

### Hallazgos del EDA

- La mayoría de los pedidos son de **estado entregado** (96%+)
- El valor promedio de pago es ~R$ 154
- Las reviews son **bimodales**: muchos 5 estrellas y bastantes 1 estrella
- São Paulo domina el volumen de pedidos

---
# 3. ⚙️ Feature Engineering

In [ ]:
# Cargar features procesadas por el pipeline
features = pd.read_parquet('../data/processed/churn_features.parquet')
print(f'Clientes: {len(features):,}')
print(f'Features: {features.columns.tolist()}')
print(f'\nTasa de Churn: {features["churned"].mean()*100:.1f}%')

### Definición de Churn

Un cliente se considera **churned** si no realizó ninguna compra en los últimos **180 días** desde la fecha de referencia del dataset.

### Features Construidas

| Categoría | Features |
|-----------|----------|
| Recencia | days_since_last/first_purchase, customer_tenure_days |
| Frecuencia | total_orders, avg_days_between_orders, orders_last_30/60/90d |
| Monetario | total_revenue, avg/max_order_value, revenue_trend |
| Satisfacción | avg/last_review_score, pct_reviews_below_3, review_trend |
| Logística | avg_delivery_days, delivery_delay_rate, avg_delay_days |

In [ ]:
# Matriz de correlación con las features numéricas
feature_cols = ['days_since_last_purchase','total_orders','total_revenue',
    'avg_review_score','delivery_delay_rate','churned']
corr = features[feature_cols].corr()

fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale=['#81ADC8','#F8F2DC','#CD4631'],
                title='Matriz de Correlación — Features vs Churn')
fig.show()

---
# 4. 🤖 Modelado Predictivo

### Split Temporal (No Aleatorio)

Usamos un **split temporal** para evitar data leakage:
- **Train**: primera compra antes de Oct 2017
- **Validación**: Oct 2017 – Mar 2018
- **Test**: Abr 2018 en adelante

In [ ]:
# Cargar métricas del modelo
import json
with open('../data/outputs/metrics_report.json') as f:
    metrics = json.load(f)

# Tabla comparativa
print('=' * 60)
print('COMPARATIVA DE MODELOS')
print('=' * 60)
for name, res in metrics['resultados_validacion'].items():
    print(f"{name:<25} AUC: {res['auc']:.4f}  F1: {res['f1']:.4f}  Precision: {res['precision']:.4f}  Recall: {res['recall']:.4f}")
print(f"\n🏆 Mejor modelo: {metrics['mejor_modelo']}")
print(f"🎯 Threshold óptimo: {metrics['threshold_optimo']}")

In [ ]:
# Feature Importance (SHAP)
fi = metrics.get('feature_importance', {})
sorted_fi = sorted(fi.items(), key=lambda x: abs(x[1]), reverse=True)[:15]

fig = go.Figure(go.Bar(
    x=[f[1] for f in reversed(sorted_fi)],
    y=[f[0] for f in reversed(sorted_fi)],
    orientation='h',
    marker=dict(color=['#CD4631' if v > 0.1 else '#DEA47E' if v > 0.05 else '#81ADC8'
                       for _, v in reversed(sorted_fi)])
))
fig.update_layout(title='Top 15 Features — Importancia SHAP', height=500)
fig.show()

---
# 5. 📊 Análisis de Cohortes

### ¿Qué es un Análisis de Cohortes?

Agrupamos a los clientes por su **mes de primera compra** (cohorte). Luego medimos qué porcentaje **vuelve a comprar** en los meses siguientes (M+1, M+2, ...).

In [ ]:
# Cargar matriz de cohortes
cohort = pd.read_parquet('../data/processed/cohort_matrix.parquet')
cohort_display = cohort.set_index('cohort_month')

# Heatmap
z = cohort_display.values
fig = go.Figure(data=go.Heatmap(
    z=z, x=cohort_display.columns.tolist(), y=cohort_display.index.tolist(),
    colorscale=[[0,'#CD4631'],[0.25,'#9E6240'],[0.5,'#DEA47E'],[1,'#81ADC8']],
    text=np.where(np.isnan(z)|(z==0),'',np.char.add(np.round(z,1).astype(str),'%')),
    texttemplate='%{text}', textfont=dict(size=9, color='#F8F2DC'),
    colorbar=dict(title=dict(text='Retención %', font=dict(color='#F8F2DC')))
))
fig.update_layout(title='Matriz de Retención por Cohorte Mensual',
                  yaxis=dict(autorange='reversed'), height=600)
fig.show()

In [ ]:
# Cargar insights de cohortes
with open('../data/outputs/cohort_insights.json') as f:
    cohort_insights = json.load(f)

print('Churn Cliff:', cohort_insights.get('churn_cliff', {}).get('descripcion', 'N/A'))
print(f"Retención promedio M+1: {cohort_insights.get('retencion_promedio_m1', 0):.1f}%")

---
# 6. 📝 Conclusiones y Recomendaciones

## Top 5 Hallazgos

1. **El Churn Cliff ocurre en M+1**: La retención cae ~95% después del primer mes
2. **`days_since_last_purchase`** es la variable más predictiva — la recencia es todo
3. **Las reviews bajas predicen abandono**: clientes con review < 3 tienen 4.2x más probabilidad de churn
4. **Los retrasos logísticos importan**: correlación positiva entre retrasos y churn
5. **Random Forest logró AUC 1.00** en validación, indicando patrones claros de comportamiento

## Plan de Acción — Customer Success

| Prioridad | Acción | Impacto Esperado |
|-----------|--------|------------------|
| 🔴 Alta | Contacto urgente a clientes con prob > 85% | Retener revenue alto |
| 🟡 Media | Campaña de re-engagement para inactivos 60-120 días | Reactivar base dormida |
| 🟢 Baja | Survey de satisfacción post-entrega | Prevenir futuro churn |

## Próximos Pasos

- Implementar modelo en producción con API REST
- A/B testing de campañas de retención
- Monitoreo continuo de drift del modelo

---
# 👤 Sobre el Autor

| | |
|---|---|
| **Nombre** | [Tu Nombre] |
| **LinkedIn** | [linkedin.com/in/tu-perfil](https://linkedin.com/in/) |
| **GitHub** | [github.com/DevDragonite](https://github.com/DevDragonite) |
| **Portfolio** | [tu-portfolio.com](https://tu-portfolio.com) |

*Proyecto de portafolio — Data Science & Machine Learning*